[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C11/E11-1_Usinage_CNC.ipynb)

# Mesures d'une broche d'usinage : reconnaître la matière à partir de treize signaux

## Le contexte

Une machine-outil à commande numérique est équipée de capteurs : vitesse de broche, avance, efforts de coupe, vibration, émission acoustique, courant et couple du moteur, température de l'outil, débit d'arrosage, puissance, rugosité obtenue et épaisseur de copeau. Trois nuances de matière passent sur la machine : un acier 42CrMo4, un alliage d'aluminium 6061 et un inox 316L. Le programme d'usinage n'indique pas toujours la nuance réellement montée, alors que les réglages en dépendent.

On voudrait reconnaître la nuance à partir des treize signaux. Ceux-ci sont nombreux et liés entre eux : l'analyse en composantes principales (ACP) les résume en quelques axes qui portent l'essentiel de leur variation. La nuance est une classe parmi trois : c'est un problème de classification.

Défi : prédire `meta_nuance_matiere` et obtenir un F-score macro supérieur à celui du modèle de départ sur la base de test. Le F-score macro est la moyenne des F-scores des trois nuances. Le modèle de départ résume les treize signaux par une seule composante principale, puis applique une régression logistique ; il atteint 0,916 : à vous de faire mieux. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Usinage_CNC.xlsx`, dans le dossier `donnees` du dépôt : 178 passes d'usinage, 14 colonnes. Une ligne est une passe.

| Colonne | Signification | Unité |
|---|---|---|
| `vitesse_broche_rpm` | vitesse de rotation de la broche | tr/min |
| `avance_mm_min` | vitesse d'avance | mm/min |
| `effort_radial_N`, `effort_axial_N` | efforts de coupe | N |
| `vibration_rms_mm_s` | vitesse de vibration efficace | mm/s |
| `emission_acoustique_dB` | niveau d'émission acoustique | dB |
| `courant_moteur_A` | courant du moteur de broche | A |
| `temperature_outil_C` | température de l'outil | °C |
| `debit_arrosage_L_min` | débit du liquide d'arrosage | L/min |
| `couple_broche_Nm` | couple sur la broche | N·m |
| `puissance_kW` | puissance absorbée | kW |
| `rugosite_Ra_um` | rugosité Ra obtenue | µm |
| `epaisseur_copeau_mm` | épaisseur du copeau | mm |
| `meta_nuance_matiere` | cible : nuance de la matière usinée | catégorie |

Origine : jeu public transformé. Les treize mesures chimiques d'un jeu public de vins sont réinterprétées, par une transformation affine, en treize signaux d'usinage, et les trois cépages en trois nuances de matière. Il enseigne une méthode, et aucune conclusion de procédé n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et l'effectif de chaque nuance : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Usinage_CNC.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Usinage_CNC.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "meta_nuance_matiere"
mesures = [colonne for colonne in df.columns if colonne != cible]
print("dimensions :", df.shape)
print("cases vides au total :", int(df.isna().sum().sum()))
print(df[cible].value_counts().to_string())
print(df[mesures].agg(["min", "max"]).T.round(1).to_string())

Lecture du résultat : 178 passes, 13 mesures, aucune case vide. Les trois nuances ne sont pas équilibrées, mais aucune n'est rare : la plus petite en compte 48. Les échelles diffèrent beaucoup : la vitesse de broche se compte en milliers de tours par minute, l'épaisseur de copeau en dixièmes de millimètre. L'ACP cherche les directions de plus grande variance : sans mise à l'échelle, elle ne verrait que les mesures à grande échelle.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Usinage_CNC.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : mettre le test de côté

Un quart des passes est mis de côté avant toute exploration croisée avec la cible. La graine 42 fixe le tirage, et la répartition des trois nuances est conservée dans les deux bases (`stratify`) : chacun obtient la même base de test, et les scores se comparent.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[mesures]
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE, stratify=y)
print("apprentissage :", len(X_train), "passes")
print("test :", len(X_test), "passes")

### Étape 3 : chercher les signaux qui évoluent ensemble

Deux signaux très corrélés portent presque la même information : c'est ce que l'ACP exploite. On compte, sur l'apprentissage seul, les paires de signaux dont la corrélation dépasse 0,5 en valeur absolue.

In [ ]:
correlations = X_train.corr(numeric_only=True)
paires = correlations.where(np.triu(np.ones(correlations.shape, dtype=bool), k=1)).stack().dropna()
fortes = paires[paires.abs() > 0.5].sort_values(key=abs, ascending=False)
print("paires de corrélation absolue supérieure à 0,5 :", len(fortes), "sur", len(paires))
print(fortes.head(4).round(2).to_string())

Lecture du résultat : 16 paires de signaux sur 78 ont une corrélation supérieure à 0,5 en valeur absolue, la plus liée étant `emission_acoustique_dB` et `courant_moteur_A` (0,85). Les signaux partagent donc une part de leur information : l'ACP les regarde tous ensemble et en tire quelques axes combinés.

### Étape 4 : calculer les composantes principales

On met les mesures à l'échelle (moyenne nulle, écart-type un), puis on calcule toutes les composantes principales de l'apprentissage. La part de variance de chaque composante dit ce qu'elle porte ; la somme cumulée dit ce que portent les premières. La seconde figure projette les passes d'apprentissage sur les deux premières composantes, avec un marqueur par nuance, ce que le modèle de départ n'utilise pas encore.

In [ ]:
# Votre code :

Lecture du résultat : la première composante porte 37,7 % de la variance des treize signaux, la deuxième 20,1 %, soit 57,8 % pour les deux. Il faut 5 composantes pour atteindre 80 %. Sur la projection, les trois nuances occupent des régions voisines mais distinctes du plan : les deux premières composantes séparent déjà l'essentiel, une seule composante en séparera moins bien.

### Étape 5 : construire le modèle de départ

Le modèle de départ résume les treize signaux par la première composante principale, puis classe la nuance par une régression logistique sur cette seule composante. Le pipeline enchaîne la mise à l'échelle, l'ACP et le classifieur, et apprend sur `X_train` seul : la mise à l'échelle et l'ACP ne voient jamais le test.

In [ ]:
# Votre code :

### Étape 6 : mesurer sur la base de test

Le modèle prédit des passes qu'il n'a jamais vues. On lit le F-score macro, l'exactitude et la matrice de confusion. Un témoin sans apprentissage, qui répond toujours la nuance la plus fréquente de l'apprentissage, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

y_pred = model.predict(X_test)
score_depart = f1_score(y_test, y_pred, average="macro")
temoin = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
score_temoin = f1_score(y_test, temoin.predict(X_test), average="macro")
print(f"F-score macro : {score_depart:.3f}")
print(f"exactitude : {accuracy_score(y_test, y_pred):.3f}")
print(f"F-score macro du témoin (toujours la nuance la plus fréquente) : {score_temoin:.3f}")
print("matrice de confusion (lignes : réel, colonnes : prédit)")
print(pd.DataFrame(confusion_matrix(y_test, y_pred, labels=model.classes_), index=model.classes_,
                   columns=model.classes_).to_string())

Lecture du résultat : le F-score macro est de 0,916, contre 0,190 pour le témoin : une seule composante, qui ne porte que 38 % de la variance, reconnaît déjà l'essentiel des nuances. La matrice de confusion compte 4 passes mal classées. La base de test compte 45 passes : chaque passe mal classée coûte plusieurs points de F-score, d'où une valeur sensible au tirage.

## Pour aller plus loin : combien de composantes, et faut-il les mettre à l'échelle ?

Le modèle de départ de l'étape 5 reste le score de référence du défi. La cellule suivante compare, par validation croisée répétée sur l'apprentissage seul, le nombre de composantes conservées, l'ACP sans mise à l'échelle, les treize signaux réduits sans ACP, l'analyse discriminante linéaire et un SVM à noyau gaussien. Le test n'est pas regardé pour choisir : il ne sert qu'une fois, au modèle retenu. Les lectures sont imprimées par la cellule elle-même.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le F-score macro du modèle de départ de l'étape 5 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = f1_score(y_test, model.predict(X_test), average="macro")
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE f1_macro = %.3f" % score)

## Synthèse

1. Les treize signaux sont liés : 16 paires sur 78 dépassent 0,5 de corrélation, et deux composantes portent 58 % de la variance.
2. L'ACP exige des signaux mis à l'échelle : sans cela, elle suit les unités et non l'information. Avec deux composantes sans mise à l'échelle, la validation croisée tombe à 0,758, contre 0,978 avec.
3. Une composante donne 0,829 en validation croisée, huit composantes 0,975 : conserver davantage de composantes aide, puis le gain s'arrête.
4. Résumer les signaux n'est pas toujours nécessaire : les treize signaux réduits, sans ACP, donnent 0,980. L'ACP sert à comprendre et à simplifier, pas à gagner en précision par elle-même.
5. Le modèle retenu par la validation croisée est « analyse discriminante linéaire » ; mesuré une seule fois sur le test, il donne 0,977. Avec 45 passes de test, ce chiffre bouge d'un tirage à l'autre.